# Basics of Cython

In [1]:
%load_ext cython
import cython

In Cython untyped dynamic var. behave exactly like Python var.

Use **cdef** to define static C vars

In [2]:
%%cython

cdef int a
cdef float b

## Integer operation don't overflow

By default, Cython checks for integer overflow when it’s doing operations that use Python semantics.
But if you tell it “integer operations don’t overflow”, Cython disables those checks — allowing faster, but potentially unsafe, arithmetic like in plain C.

In [12]:
@cython.infer_types(True) # It enables type inference — Cython’s ability to guess the right C type from context.

def infer_dec():
    a = 1
    b = 2.0
    c = 3 + 4j
    r = a*b + c
    return r

In [15]:
%%cython

# cython: overflowcheck=True
cdef int a = 2147483647
a += 1   # Raises OverflowError

# cython: overflowcheck=False
cdef int b = 2147483647
b += 1   # Wraps to -2147483648 (C behavior)


Dereferencing in Cython is different than in C.

In [31]:
%%cython

cdef double hot_pizza
cdef double *topping

topping = &hot_pizza      # topping now points to the memory address of hot_pizza
topping[0] = 1.618        # change the value at that memory location

print(hot_pizza)
print(topping[0])
# => 1.618

1.618
1.618


Alternatively you can use 'cython.operator.dereference'

In [26]:
%%cython

cdef int a,b,c
tuple_of_ints = (a,b,c)

You can also use **cdef** to define static Python vars

In [30]:
%%cython

cdef list postal_codes, modified_postal_codes
cdef dict names_from_postal_codes
cdef str pname
cdef set unique_postal_codes

names_from_postal_codes = {"1000": "Berlin", "2000": "Hamburg"} # initialization
postal_codes = list(
    names_from_postal_codes.keys()
    )

**Division&Modulus** semantics are different in Python and C. Cython uses Python semantics by default.

Static typing allows Cython to remove dynamic dispatch on static_recipes

Cython supports Python types like:
_complex, type/object, array, slice,_ etc.

## C functions
* C functions can be put as an argument to other C functions but cannot be defined inside other C functions.
* C functions are statically assigned to a name
* C functions take arguments only as positions. (?)
* Though Python functions are more flexible, they are slower.

## Standard Libraries
### 1. Distutils
* Gives full control to compile Cython codes
* 'cythonize' command compiles to C/C++
* Control the behavior of distutils through a Python script, typically named 'setup.py'

In [99]:
# minimal setup.py file for test.pyx

from distutils.core import setup
from cython.build import cythonize

setup(
    ext_modules=cythonize('test.pyx')
)

ModuleNotFoundError: No module named 'cython.build'; 'cython' is not a package

### 2. IPython
* Allows to write Cython directly in IPython interpreter. (?)
* Use '%' sign to tell IPython to load Cython magic commands. (Ex: %%cython)

In [98]:
%%cython

def fib(int n):
    cdef int i
    cdef double a=0.0, b=1.0
    for i in range(n):
        a, b = a+b, a

### 3. pyximport
* retrofits the import statement to recognize '.pyx' extension modules, send automatically and import the extension module by Python
* BASE_NAME.pyx file, we have BASE_NAME.pyxdeps

### How Python functions work in Cython? 

In [46]:
%%cython
# cy_func.pyx

def factorial(num):
    if num <= 1:
        return 1
    return num * factorial(num-1)

# ipython
# >>> import pyximport
# >>> import pyximport.install()
# >>> import cy_func
# >>> cy_func.factorial?
# >>> cy_func.factorial(10)

### How C functions work in Cython?

use cdef functions

In [65]:
%%cython

cdef long c_factorial(long num):
    if num <= 1:
        return 1
    return num * c_factorial(num-1)

print(c_factorial(5))

def c_wrap_factorial(n):
    "Python Wrapper"
    return c_factorial(n)
    # unfortunately, it gives incorrect results for large n values because Python objects & C types don't always map to each other perfectly.

print(c_wrap_factorial(5))

120
120


Cython doesn't allow cdef function be called from external python code. cdef function used as fast supporting function.

## Hybrid function

use cpdef

In [67]:
%%cython

cpdef long cp_factorial(long num):
    if num <= 1:
        return 1
    return num * cp_factorial(num-1)

Of course **cpdef** functions have limitations. They must be compatible with both Python & C types. Not all C types can be represented in Python.

## Loops in Cython

* There are some guidelines for Cython to produce efficient loops:
    
    1. When looping over a range call, type range argument as C integer. Cython automatically type the loop _index var i_ as an _int_.

    2. When looping over container statically typing loop indexing variable may introduce more overhead. (?) To tackle this problem, consider converting the container to a **C++** equivalent container.

    3. For efficient while-liios make loop condition expression efficient. (?) While-True loops with an internal break are efficiently translated to C automatically.

In [97]:
%%cython

a = [1.0, 5.0, 6.0, 1.0]

cdef unsigned int i, n = len(a) - 1

for i in range(1,n):
    a[i] = (a[i-1] + a[i] + a[i+1])/3.0

a; print(a)

[1.0, 4.0, 3.6666666666666665, 1.0]


# QDMI IMPLEMENTATION
## Implement some functions

In [19]:
import os
os.chdir('/workspaces/MQSS-QDMI-Devices-Suite/src/simulators/eviden_cython')
print(os.getcwd())


/workspaces/MQSS-QDMI-Devices-Suite/src/simulators/eviden_cython


In [4]:
# qaptiva_auxiliary.py

def submit_job(remote_qpu, qasm_string, nshots):
    """! Submits a quantum job to a remote QPU.

    @param remote_qpu A RemoteQPU instance.
    @param qasm_string QASM code string.
    @param nshots Number of shots to run the job.
    @return A list containing measurement states and their probabilities, or None if submission fails.
    """
    try:
        from qat.interop.openqasm import OqasmParser
        parser = OqasmParser()
        circuit = parser.compile(qasm_string)
        job = circuit.to_job(nbshots=nshots)
        raw_results = remote_qpu.submit(job)
        states = []
        probabilities = []
        for result in raw_results:
            states.append(result.state.bitstring)
            probabilities.append((result.probability))
        return_value = [",".join(states)] + list(probabilities)
        return return_value
    except:
        return None

In [32]:
%%cython

cdef extern from "/workspaces/MQSS-QDMI-Devices-Suite/src/simulators/eviden_cython/constants.h":
    ctypedef enum QDMI_Status:
        QDMI_SUCCESS
        QDMI_ERROR_INVALIDARGUMENT
        QDMI_ERROR_FATAL

    ctypedef int QAPTIVA_QDMI_DEVICE_SESSION_STATUS
    ctypedef int QDMI_Job_Status
    ctypedef int QDMI_Program_Format

    cdef struct QAPTIVA_QDMI_Device_Session_impl_d:
        char* url
        int* is_noisy_session
        QAPTIVA_QDMI_DEVICE_SESSION_STATUS status

    cdef struct QAPTIVA_QDMI_Device_Job_impl_d:
        QAPTIVA_QDMI_Device_Session_impl_d* session # pointer explicitly
        int id
        QDMI_Job_Status status
        size_t* num_shots
        QDMI_Program_Format* format
        char* program
        double* probability_dense
        char* probability_keys
        double* probability_values
        size_t* n_state
        size_t* results_size
        double* t1
        double* t2

In [34]:
%%cython

cpdef object create_remote_qpu(str host):
    """
    Creates a remote QPU connection.

    Parameters
    ----------
    host : str
        Hostname and port in the form "host:port".

    Returns
    -------
    object
        A RemoteQPU instance, or None if connection fails.
    """
    cdef object RemoteQPU, qpu
    cdef str url
    cdef str port

    try:
        from qat.core.qpu import RemoteQPU
        url, port = host.split(":")
        # Convert port to int if the API expects it
        qpu = RemoteQPU(int(port), url)
        return qpu
    except Exception:
        return None

In [ ]:
%%cython


from libc.math cimport isnan
cimport cython

@cython.cfunc
@cython.inline
cdef bint _is_bad(double x) nogil:
    return x <= 0 or isnan(x)

cpdef object submit_noisy_job(str host, str qasm_string, int nshots, double t1=40000, double t2=22000):
    """
    Submit a noisy job via HTTP to a Flask backend.

    Returns:
        ["state1,state2,...", p1, p2, ...] on success
        None on handled HTTP errors

    Raises:
        ValueError on invalid inputs
        requests.RequestException on network errors (after printing message)
    """
    cdef dict payload
    cdef object requests, resp, data, result, probs
    cdef list states, probabilities

    # ---- validate inputs (client-side) ----
    if nshots <= 0:
        raise ValueError("nshots must be a positive integer")
    if _is_bad(t1) or _is_bad(t2):
        raise ValueError("t1 and t2 must be positive floats")

    payload = {
        "aqasm": qasm_string,
        "t1": t1,
        "t2": t2,
        "nbshots": nshots,
    }

    try:
        import requests
        resp = requests.post(host, json=payload, timeout=10)
        if resp.status_code != 200:
            # Server returned an application error; surface and return None
            try:
                print("Error from server:", resp.text)
            except Exception:
                pass
            return None

        data = resp.json()
        result = data.get("result", {})
        probs = result.get("state_probabilities", {})

        # Expect a mapping: { "010": 0.12, "111": 0.88, ... }
        if not isinstance(probs, dict):
            raise ValueError("Malformed server response: 'state_probabilities' must be a dict")

        # For deterministic order, sort states lexicographically
        states = sorted(probs.keys())
        probabilities = [probs[s] for s in states]

        return [",".join(states)] + probabilities

    except Exception as e:
        # Network/JSON/parsing issues -> re-raise so caller can handke
        print("HTTP submit error:",e)
        raise


Error compiling Cython file:
------------------------------------------------------------
...
        states = sorted(probs.keys())
        probabilities = [probs[s] for s in states]

        return [",".join(states)] + probabilities

        except Exception as e:
        ^
------------------------------------------------------------

/root/.cache/ipython/cython/_cython_magic_12a42236736e41ead2cca7f1ccfc54dc3b6c3875a12b485c09e1ebd9afd57cb8.pyx:65:8: Expected an identifier or literal


In [33]:
%%cython

cpdef object submit_job(object remote_qpu, str qasm_string, int nshots):
    """
    Submits a quantum job and returns:
        ["state1,state2,...", p1, p2, ...]
    or None on failure.
    """
    cdef object OqasmParser, parser, circuit, job, raw_results, r
    cdef list states = []
    cdef list probs = []

    try:
        from qat.interop.openqasm import OqasmParser
        parser = OqasmParser()
        circuit = parser.compile(qasm_string)
        job = circuit.to_job(nbshots=nshots)
        raw_results = remote_qpu.submit(job)

        for r in raw_results:
            # Assuming result has .state.bitstring and .probability
            states.append(r.state.bitstring)
            probs.append(r.probability)

        return [",".join(states)] + probs
    except Exception:
        # You can `print(e)` or log here if you want visibility
        return None